In [28]:
TARGET_NAME="LeNet-300-100"
TARGET_DIR="../results/1.LeNet-300-100"
TARGET_DATE="20260402"
TARGET_TYPE="canonical"
MODEL_ID1=1
MODEL_ID2=3
TARGET_CONFIG=f"freq_{MODEL_ID1}_{MODEL_ID2}.json"

In [29]:
files = [
    f"{TARGET_DIR}/package_header_{MODEL_ID1}_{MODEL_ID2}.bin",
    f"{TARGET_DIR}/package_symbol_table_{MODEL_ID1}_{MODEL_ID2}.bin",
    f"{TARGET_DIR}/package_payload_{MODEL_ID1}_{MODEL_ID2}.bin",
]

In [30]:
with open(
     f"{TARGET_DIR}/package_header_{MODEL_ID1}_{MODEL_ID2}.bin", "rb"
) as f:
    header = f.read()

print(type(header))
print(len(header))  # size in bytes
for i in range(0, len(header), 8):
    chunk = header[i:i+8]
    print(' '.join(f'{b:02X}' for b in chunk))

<class 'bytes'>
48
51 2D 48 55 46 44 45 4C
00 00 01 00 03 00 06 01
04 0C 08 09 3E FF 00 00
EE 09 04 9F D6 8F A2 7D
72 58 03 F4 80 84 25 CA
80 21 04 00 60 D9 1C 00


In [31]:
chunks = []

for fname in files:
    with open(fname, "rb") as f:
        while chunk := f.read(4096):   # walrus operator, Python 3.8+
            chunks.append(bytearray(chunk))

package = bytearray().join(chunks)

assert len(package) % 8 == 0, "Package must be a multiple of 8 bytes"

In [32]:
len(package)

236688

In [33]:
for i in range(0, 80, 8):
    chunk = package[i:i+8]
    print(' '.join(f'{b:02X}' for b in chunk))

51 2D 48 55 46 44 45 4C
00 00 01 00 03 00 06 01
04 0C 08 09 3E FF 00 00
EE 09 04 9F D6 8F A2 7D
72 58 03 F4 80 84 25 CA
80 21 04 00 60 D9 1C 00
01 00 21 1A 12 11 54 35
1E 00 70 A2 51 18 84 3E
1D 0D 86 42 E1 50 98 44
1E 0D 05 82 40 E0 50 10


In [34]:
import struct
import zlib

def update_crc32(data: bytes) -> bytes:
    """
    Calculate CRC-32 of all bytes except the last 4,
    then replace the last 4 bytes with the calculated CRC-32.
    
    Args:
        data: Input byte array (must be at least 4 bytes long)
    
    Returns:
        New byte array with last 4 bytes replaced by CRC-32
    """
    if len(data) < 4:
        raise ValueError("Data must be at least 4 bytes long")
    
    # Calculate CRC-32 of everything except the last 4 bytes
    payload = data[:-4]
    crc = zlib.crc32(payload) & 0xFFFFFFFF  # Ensure unsigned 32-bit integer

    # Pack CRC as little-endian unsigned 32-bit integer
    crc_bytes = struct.pack('<I', crc)

    # Replace the last 4 bytes with the CRC
    return payload + crc_bytes


In [35]:
package = update_crc32(bytes(package))
# Verify
payload = package[:-4]
crc_stored = struct.unpack('<I', package[-4:])[0]
crc_check  = zlib.crc32(payload) & 0xFFFFFFFF
print(f"CRC-32   : {crc_stored:#010x}")
print(f"Verified : {crc_stored == crc_check}")

CRC-32   : 0x5fa8ce43
Verified : True


In [36]:
with open(f"{TARGET_DIR}/q-hufdel_{TARGET_NAME}_{MODEL_ID1}_to_{MODEL_ID2}.bin", "wb") as f:
    f.write(package)